In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import joblib

# ============================================================
# 1. DEFINE INPUT AND OUTPUT PATHS
# ============================================================

raw_base = "../data_raw/nasa_battery"
metadata_path = "../data_raw/nasa_battery/cleaned_dataset/metadata.csv"
base_data = "../data_raw/nasa_battery/cleaned_dataset/data"

output_dir = "../data_processed/NASA"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# 2. CHECK THE ORIGINAL DATASET STRUCTURE
# ============================================================

for root, dirs, files in os.walk(raw_base):
    print(root)
    for f in files:
        print("  ", f)

# ============================================================
# 3. LOAD THE METADATA FILE
# ============================================================

meta = pd.read_csv(metadata_path)

print(meta.head())
print(meta.columns)

# ============================================================
# 4. SELECT DISCHARGE CYCLES ONLY
# ============================================================

meta_dis = meta[meta["type"] == "discharge"].copy()
print(meta_dis.head())

# ============================================================
# 5. PROCESS CAPACITY VALUES AND LABEL ANOMALOUS CYCLES
# ============================================================

meta_dis["Capacity"] = (
    meta_dis["Capacity"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.strip()
)

meta_dis["Capacity"] = pd.to_numeric(meta_dis["Capacity"], errors="coerce")
meta_dis = meta_dis.dropna(subset=["Capacity"]).reset_index(drop=True)

capacity_threshold = meta_dis["Capacity"].quantile(0.20)
meta_dis["y_true_file"] = (meta_dis["Capacity"] < capacity_threshold).astype(int)

print("Capacity threshold:", capacity_threshold)
print(meta_dis["y_true_file"].value_counts())

# ============================================================
# 6. CREATE THE CYCLE DISTRIBUTION TABLE
# ============================================================

split_counts = meta_dis["y_true_file"].value_counts().sort_index()
normal_files = int(split_counts.get(0, 0))
anomaly_files = int(split_counts.get(1, 0))
total_files = normal_files + anomaly_files

split_df = pd.DataFrame({
    "Trieda": ["Normálne cykly", "Anomálne cykly", "Spolu"],
    "Počet súborov": [normal_files, anomaly_files, total_files],
    "Podiel (%)": [
        round(normal_files / total_files * 100, 2) if total_files > 0 else 0,
        round(anomaly_files / total_files * 100, 2) if total_files > 0 else 0,
        100.00
    ]
})

print("\nRozdelenie cyklov podľa Capacity:")
print(split_df)

split_table_path = os.path.join(output_dir, "nasa_capacity_split_table.csv")
split_df.to_csv(split_table_path, index=False)

# ============================================================
# 7. SAVE THE CYCLE DISTRIBUTION TABLE AS AN IMAGE
# ============================================================

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.axis("off")

table = ax.table(
    cellText=split_df.values,
    colLabels=split_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#fde9d9")
    else:
        cell.set_facecolor("#f9f9f9")

plt.title("Rozdelenie vybíjacích cyklov na normálne a anomálne", fontsize=13)

split_table_img_path = os.path.join(output_dir, "nasa_capacity_split_table.png")
fig.savefig(split_table_img_path, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 8. LOAD ALL DISCHARGE CYCLE FILES
# ============================================================

dfs = []

for _, row in meta_dis.iterrows():
    fname = row["filename"]
    path = os.path.join(base_data, fname)

    if os.path.exists(path):
        df = pd.read_csv(path)
        df["source_file"] = fname
        df["Capacity"] = row["Capacity"]
        df["y_true_file"] = row["y_true_file"]
        dfs.append(df)

data_all = pd.concat(dfs, ignore_index=True)

print(data_all.head())
print(data_all.info())
print(data_all.describe())

# ============================================================
# 9. CREATE BASIC FEATURES
# ============================================================

num_cols = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured",
    "Current_load",
    "Voltage_load",
    "Time"
]

features = data_all[num_cols].copy()
features["Capacity"] = data_all["Capacity"]
features["y_true_file"] = data_all["y_true_file"]
features["source_file"] = data_all["source_file"]

print(features.head())

# ============================================================
# 10. NORMALIZE NUMERICAL FEATURES AND SAVE THE SCALER
# ============================================================

scaler = StandardScaler()
X = scaler.fit_transform(features[num_cols])

features_path = os.path.join(output_dir, "nasa_battery_features.csv")
scaler_path = os.path.join(output_dir, "nasa_battery_scaler.pkl")

features.to_csv(features_path, index=False)
joblib.dump(scaler, scaler_path)

print("Saved features + scaler")

# ============================================================
# 11. CREATE ROLLING FEATURES FOR EACH DISCHARGE CYCLE
# ============================================================

window = 50
rolling_parts = []

for fname, group in data_all.groupby("source_file"):
    group = group.sort_values("Time").reset_index(drop=True)

    feat_part = pd.DataFrame({
        "source_file": group["source_file"],
        "Capacity": group["Capacity"],
        "y_true_file": group["y_true_file"],

        "Voltage_measured": group["Voltage_measured"],
        "Current_measured": group["Current_measured"],
        "Temperature_measured": group["Temperature_measured"],
        "Current_load": group["Current_load"],
        "Voltage_load": group["Voltage_load"],
        "Time": group["Time"],

        "V": group["Voltage_measured"],
        "I": group["Current_measured"],
        "T": group["Temperature_measured"],

        "V_diff1": group["Voltage_measured"].diff(),
        "I_diff1": group["Current_measured"].diff(),
        "T_diff1": group["Temperature_measured"].diff(),

        "V_roll_mean": group["Voltage_measured"].rolling(window).mean(),
        "V_roll_std": group["Voltage_measured"].rolling(window).std(),
        "V_roll_min": group["Voltage_measured"].rolling(window).min(),
        "V_roll_max": group["Voltage_measured"].rolling(window).max(),

        "T_roll_mean": group["Temperature_measured"].rolling(window).mean(),
        "T_roll_std": group["Temperature_measured"].rolling(window).std(),

        "I_roll_mean": group["Current_measured"].rolling(window).mean(),
        "I_roll_std": group["Current_measured"].rolling(window).std(),
    })

    feat_part = feat_part.dropna().reset_index(drop=True)
    rolling_parts.append(feat_part)

feat = pd.concat(rolling_parts, ignore_index=True)

print(feat.head())

rolling_features_path = os.path.join(output_dir, "nasa_battery_features_rolling.csv")
feat.to_csv(rolling_features_path, index=False)

print("Saved rolling features")

# ============================================================
# 12. DISPLAY A SAMPLE OF THE BASIC FEATURES TABLE
# ============================================================

df_nasa = pd.read_csv(features_path)

print(df_nasa.columns)

df_show = df_nasa[[
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured",
    "Time",
    "Capacity",
    "y_true_file"
]].copy()

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")

table = ax.table(
    cellText=df_show.head().round(3).values,
    colLabels=df_show.columns,
    loc="center"
)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#fde9d9")
    else:
        cell.set_facecolor("#f9f9f9")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)

plt.title("Ukážka multivariantných senzorových dát batérie počas vybíjania", fontsize=14)

nasa_table_path = os.path.join(output_dir, "nasa_table.png")
fig.savefig(nasa_table_path, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 13. PLOT NORMAL AND ANOMALOUS DISCHARGE CYCLES
# ============================================================

normal_file = meta_dis[meta_dis["y_true_file"] == 0]["filename"].iloc[0]
anomaly_file = meta_dis[meta_dis["y_true_file"] == 1]["filename"].iloc[0]

plot_normal = data_all[data_all["source_file"] == normal_file].copy()
plot_anomaly = data_all[data_all["source_file"] == anomaly_file].copy()

plt.figure(figsize=(12, 4))
plt.plot(plot_normal["Time"].values, plot_normal["Voltage_measured"].values)
plt.title("Príklad normálneho vybíjacieho cyklu batérie")
plt.xlabel("Čas")
plt.ylabel("Napätie")
plt.grid(True)
plt.tight_layout()

normal_cycle_path = os.path.join(output_dir, "nasa_normal_cycle.png")
plt.savefig(normal_cycle_path, dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(plot_anomaly["Time"].values, plot_anomaly["Voltage_measured"].values)
plt.title("Príklad anomálneho vybíjacieho cyklu batérie")
plt.xlabel("Čas")
plt.ylabel("Napätie")
plt.grid(True)
plt.tight_layout()

anomaly_cycle_path = os.path.join(output_dir, "nasa_anomaly_cycle.png")
plt.savefig(anomaly_cycle_path, dpi=300, bbox_inches="tight")
plt.show()